# **综合实践：MaxPool算子的SIMT编程与混合编程实现**

## 概述

本小节是第七章SIMT编程（07.05）与SIMD/SIMT混合编程（07.06）两节的综合实践。前两节以Transpose算子为主线，带你走完了一条完整的优化路径；本节换一个算子——MaxPool（最大池化），请你**独立完成从最朴素实现到最终优化版本的全过程**，并在两种编程模式下分别实现，最后用实测数据对比两条路径。

与前两节课后小习题的区别在于：小习题只要求你在给定骨架里补全一两处 `TODO`；**本节不提供填空式骨架**，只给两份"host已写好、核函数留空"的框架代码，每个任务的正文只描述实现方案，核函数要你自己从零写出来。五个任务串成一条完整的优化链路——每个任务都建立在上一个任务的基础上，你需要自己判断每一步优化是否真的带来收益，并解释为什么。

### 学习前置要求

- 已完成07.05《SIMT Transpose算子优化实践》，掌握 `blockIdx`/`threadIdx`、`__ubuf__` UB声明、`asc_syncthreads()`、grid-stride循环、双缓冲。
- 已完成07.06《SIMD与SIMT混合编程实践》，掌握 `__global__ __vector__` 核函数、`asc_vf_call` 调起SIMT VF、`asc_copy_gm2ub_align`/`asc_copy_ub2gm_align` 搬运、`asc_lock`/`asc_unlock` 管理 `PIPE_MTE2`/`PIPE_V`/`PIPE_MTE3` 流水。
- 了解基本的Ascend C算子开发与执行流程。

### 学习目标

完成本作业后，你应该能够：

- 独立完成一个算子从朴素实现到多级优化的完整开发流程，而不是照着骨架填空。
- 判断一项优化手段（warp分段规约、UB中转、限制Thread Block数、双缓冲）在特定算子上**是否适用**，并用实测数据支撑判断——包括识别出"优化反而变慢"和"优化没有收益"的情况并解释原因。
- 理解优化之间的**相互影响**：两项针对同一份开销的优化收益不会叠加，因此一项优化的"收益百分比"强烈依赖基线选在哪里。
- 对比SIMT编程 与SIMD/SIMT混合编程两种模式在同一算子上的实现差异与性能差异，理解MTE搬运单元的价值。
- 建立"先看访存模式、再选优化手段"的调优习惯，并学会用流水分项数据确认开销是被消除了还是只是转移到了别处。

### 本节内容

- 环境准备
- MaxPool算子功能介绍与作业任务总览
- 任务1~3：SIMT编程 实现路径（含warp分段规约优化访存形态）
- 任务4~5：SIMD/SIMT混合编程实现路径（MTE搬运 + UB中转 + 双缓冲）
- 结果对比与作业提交要求

## 1. 环境准备

正式开始之前，先执行下方脚本检查CANN Toolkit是否可用，并把CANN环境变量加载到当前Jupyter进程，保证后续能够正常编译和运行算子。

本节所有编译、运行和练习修改都在 `Sources/07.07` 目录下进行，`src` 目录仅作为只读的骨架代码仓库。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

WORKSPACE = Path("Sources/07.07")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f"CANN Toolkit: {cann_home}")
print(f"Workspace: {WORKSPACE.resolve()}")


## 2. MaxPool算子功能介绍

MaxPool（最大池化）是一种常见的下采样操作，把输入矩阵划分成互不重叠的窗口，每个窗口内取最大值作为输出。本作业使用 `4 x 4` 窗口、步长为 `4`（不重叠），计算公式为：

```text
output(i, j) = max(input[4i:4i+4, 4j:4j+4])
```

也就是说，输出矩阵中的每个元素，是输入矩阵中对应 `4 x 4` 窗口内16个元素的最大值。由于窗口不重叠，且 `1024 / 4` 能整除，不需要考虑边界padding。

**算子规格：**

| 项 | 取值 |
| --- | --- |
| 输入 `input` | `(1024, 1024)`，`float` |
| 输出 `output` | `(256, 256)`，`float` |
| 窗口/步长 | `4 x 4`，不重叠 |

### 2.1 band的概念：本作业的核心切分单位

理解本作业的关键在于 **band**（条带）这个切分单位：把输入矩阵按每 `4` 行（正好是一个窗口的高度）划分成一个band，一个band的大小是 `4 x 1024`。

一个band经过池化后，恰好对应输出矩阵的**一整行**（`1024 / 4 = 256` 个元素）：

```text
输入 band 0（第 0~3 行，4 x 1024）   ->  输出第 0 行（256 个元素）
输入 band 1（第 4~7 行，4 x 1024）   ->  输出第 1 行（256 个元素）
...
输入 band 255（第 1020~1023 行）     ->  输出第 255 行
```

因此总共有 `1024 / 4 = 256` 个band，与输出矩阵的256行一一对应。

按band切分带来一个重要性质：**MaxPool的输入读取和输出写回在GM上天然都是连续的**。一个band在GM上就是 `4 * 1024` 个连续的 `float`；它对应的输出是 `256` 个连续的 `float`。band内读取、band输出写回都不需要跨行跳跃。

这一点和Transpose形成鲜明对比：Transpose的输入按行连续、输出必须按列连续写回，这个矛盾来自转置操作本身，无法消除，只能把非连续访问转移到UB内部。**而MaxPool从第一步实现开始就不存在非连续访问的问题。** 请在做后续任务时始终记住这个差异——它会直接决定各项优化手段是否有效。


### 2.2 作业任务总览

本作业分为两条路径、共五个任务。前三个任务是SIMT编程 实现路径（直接基于GM读写），后两个任务是SIMD/SIMT混合编程实现路径（引入MTE搬运和UB中转）：

| 任务 | 目录 | 核函数 | 本步引入的变化 |
| --- | --- | --- | --- |
| 任务1 | `gm_naive` | `maxpool_gm_naive_custom` | SIMT编程，直接基于GM读写，Thread Block数 = band数 = 256 |
| 任务2 | `gm_core_limited` | `maxpool_gm_core_limited_custom` | Thread Block数限制为物理核数，核内grid-stride循环 |
| 任务3 | `gm_warp_shfl` | `maxpool_gm_warp_shfl_custom` | 用 `asc_shfl_down` 做warp内分段规约，改造线程映射与访存形态 |
| 任务4 | `simd_simt_ub` | `maxpool_simd_simt_ub_custom` | 混合编程：MTE搬运 + UB中转（单缓冲），VF沿用任务3的规约方式 |
| 任务5 | `simd_simt_ub_db` | `maxpool_simd_simt_ub_db_custom` | 混合编程 + 双缓冲，三条流水线重叠 |

任务顺序的安排有意区分了两类优化：**任务3 优化的是"算得怎么样"（SIMT VF内部的线程映射与访存形态），任务4/5 优化的是"数据怎么来"（GM/UB之间的搬运方式与流水重叠）**。

**关于骨架代码**：本作业**不逐任务给出填空式骨架**，只提供两份"host已写好、核函数留空"的框架代码，放在只读目录 `src/07_07_maxpool/` 下：

| 框架 | 适用任务 | 编译选项 | 已写好的部分 | 留给你的部分 |
| --- | --- | --- | --- | --- |
| `simt_skeleton/maxpool_simt.asc` | 任务1、2、3 | 带 `--enable-simt` | 数据构造、golden、校验、ACL流程、核数查询、启动配置的位置 | 核函数全部内容 + 启动的 grid/block |
| `hybrid_skeleton/maxpool_hybrid.asc` | 任务4、5 | **不带** `--enable-simt` | 同上 | 搬运函数、SIMT VF、核函数全部内容 |

两份框架而不是一份，是编译器的硬性约束：`asc_simd.h` 里有一条 `#error`，禁止和 `--enable-simt` 同时使用。所以纯SIMT 和 SIMD/SIMT混合编程两种模式必须用不同的编译选项，也就必须是两份工程。

框架代码原样编译可以通过，但核函数是空的、不写出任何结果，因此校验会失败——这是预期状态，方便你先跑通编译链路再动手实现。

**每个任务的正文只描述实现方案**（线程如何映射、每一步做什么、有哪些约束和可用接口），不给出可直接填空的代码位置。请据此自己写出核函数。写不下去时再看该任务末尾的参考答案。

参考答案是**每个任务一份完整可独立编译的程序**（放在 `answer/07_07_maxpool/` 下），核函数用了各自的专有名字（如 `maxpool_gm_naive_custom`），而框架里的核函数叫 `maxpool_custom`——这只是命名差异，对照时不用在意。你也可以直接改框架里的名字，只要核函数定义、`__launch_bounds__` 和host侧的启动调用三处保持一致。

执行下面的单元格，把两份框架代码分发到 `Sources/07.07/maxpool/` 下的五个任务目录（每个任务一份独立副本，这样五个版本可以同时保留、方便最后做性能对比）：

In [ ]:
import shutil
from pathlib import Path

SRC_ROOT = Path("src/07_07_maxpool")            # 只读框架代码目录
DST_ROOT = Path("Sources/07.07/maxpool")        # 工作目录

# 每个任务一份独立副本：任务1~3 用纯SIMT 框架，任务4~5 用混合编程框架。
# 独立副本的目的是让五个版本可以同时保留，最后一起做性能对比。
TASKS = {
    "task1_gm_naive":       ("simt_skeleton",   "maxpool_simt.asc"),
    "task2_gm_core_limited": ("simt_skeleton",   "maxpool_simt.asc"),
    "task3_gm_warp_shfl":   ("simt_skeleton",   "maxpool_simt.asc"),
    "task4_simd_simt_ub":   ("hybrid_skeleton", "maxpool_hybrid.asc"),
    "task5_simd_simt_ub_db": ("hybrid_skeleton", "maxpool_hybrid.asc"),
}

# 清理旧的工作目录，保证每次都从 src 拷贝出干净的一份
# 注意：如果你已经在 Sources 下写了代码，重新执行本 cell 会覆盖掉你的修改
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

for task, (skeleton, asc_name) in TASKS.items():
    dst = DST_ROOT / task
    dst.mkdir(parents=True, exist_ok=True)
    for pattern in ("*.asc", "*.h", "CMakeLists.txt"):
        for f in sorted((SRC_ROOT / skeleton).glob(pattern)):
            shutil.copy2(f, dst / f.name)
    print(f"{task}: {skeleton} -> {sorted(p.name for p in dst.iterdir())}")

## 3. 任务1：SIMT编程 直接基于GM读写

### 3.1 实现方案

先实现最朴素的版本：不使用任何UB中转，每个线程直接从GM读取自己负责的 `4 x 4` 窗口共16个元素，做最大值规约后直接写回GM。

**线程映射**

- 一个Thread Block负责一个band，因此Thread Block数 = band数 = `256`。
- 一个Thread Block启动 `256` 个线程（等于输出矩阵的宽度），`threadIdx.x` 即本线程负责的输出列。
- 本线程负责的窗口在输入矩阵中的位置：行范围 `[band_id * 4, band_id * 4 + 4)`，列范围 `[out_col * 4, out_col * 4 + 4)`。

**核函数要做的事**

用两层循环遍历自己窗口内的16个元素，逐个比较取最大值，把结果写到 `output[band_id * out_width + out_col]`。一个线程一个输出，不需要任何线程间同步。

**Host侧要改的地方**

框架代码里 `dim3 grid` / `dim3 block` 的位置有注释标出。本任务 `grid = total_bands`（256）、`block = out_width`（256）。核函数的 `__launch_bounds__` 要和实际启动的线程数一致。

**两个实现细节**

- 规约初值建议直接取窗口左上角元素，而不是取 `0` 或某个极小常量——输入数据可能全为正也可能含负数，用窗口内的真实元素做初值最稳妥。
- 请用三目比较逐元素累积（`max_val = (v > max_val) ? v : max_val;`），不要使用 `std::max`/`fmaxf`，保持与host侧golden函数一致的比较行为。

### 3.2 编写并编译运行

框架代码在 `Sources/07.07/maxpool/task1_gm_naive/maxpool_simt.asc`。执行下面的cell可以查看它——host部分已经写好，你需要填的是空的核函数和启动配置：

三份纯SIMT 任务（任务1、2、3）用的是同一份框架，因此这份代码在任务2、3中也是起点。

In [ ]:
!cat Sources/07.07/maxpool/task1_gm_naive/maxpool_simt.asc

实现后执行下面的单元格编译运行：

In [ ]:
!cd Sources/07.07/maxpool/task1_gm_naive && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

实现正确时，输出中会包含校验通过信息：

```text
<你实现的功能正确性验证结果>
[Success] Case accuracy verification passed.
```

**参考答案：**

In [ ]:
!cat answer/07_07_maxpool/gm_naive/maxpool_gm_naive.asc

## 4. 任务2：限制Thread Block数为物理核数

### 4.1 实现方案

任务1让Thread Block数等于band数（256），而设备上只有64个物理vector core，因此存在4倍**超发**：同一时刻只有64个Block在执行，其余的排队等待。本任务把Thread Block数固定为物理核数，改用**grid-stride循环**在核内处理多个band。

**要改的地方**

- Host侧：`grid = num_blocks`（物理核数，框架里已经用 `get_vector_core_num()` 查好并打印），`block` 不变仍是 `out_width`。
- 核函数：外面套一层grid-stride循环——`band_id` 从 `blockIdx.x` 开始，每次步进 `gridDim.x`，直到超过 `total_bands`。循环体内部的窗口规约逻辑和任务1**完全相同**，一行都不用改。

grid-stride的写法是SIMT编程里的标准模式，值得记牢：它把"Block数"和"任务数"解耦，同一份核函数在任意Block数下都能正确覆盖全部数据。

**请在运行前先预测**：消除4倍超发的排队开销，你认为耗时会降多少？

### 4.2 编写并编译运行

框架代码与任务1相同（`Sources/07.07/maxpool/task2_gm_core_limited/maxpool_simt.asc`），你可以把任务1的实现拷过来再加grid-stride循环。

In [ ]:
!cd Sources/07.07/maxpool/task2_gm_core_limited && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

### 4.3 参考答案

In [ ]:
!cat answer/07_07_maxpool/gm_core_limited/maxpool_gm_core_limited.asc


#### 4.3.1 性能数据采集与分析
请测试性能数据分析优化效果。

以下为Ascend 950 环境、CANN 9.2.0 上实测（`msopprof` Task Duration，5次取均值，64物理核）的参考结果：

| 任务 | 核函数 | Task Duration(μs) | Thread Block数 |
| --- | --- | --- | --- |
| 任务1 | `maxpool_gm_naive_custom` | 13.00 | 256 |
| 任务2 | `maxpool_gm_core_limited_custom` | 14.04 | 64 |

**这里出现了第一个和Transpose不同的结论**：限制Thread Block数**不但没有带来收益，反而略微变慢了**（13.00 → 14.04μs，慢约8%）。回忆07.05节，同样这一步让Transpose从57.37μs降到35.674μs，收益接近40%。为什么MaxPool上完全是另一个结果？

原因在于两个算子的**任务规模差异**。Transpose的输入输出都是 `1024 x 1024`，需要读4MB、写4MB；MaxPool读4MB但只写 `256 x 256`（256KB），且每16个输入元素才产出1个输出。MaxPool的整体耗时只有十几μs，比Transpose优化前的57μs小一个数量级。在这个量级下，Thread Block调度排队的头尾开销本身就已经不是瓶颈了——超发4倍（256/64）带来的排队开销，相对于访存耗时可以忽略。


不过，任务2的grid-stride循环结构本身在后续任务中是**必需**的：任务4/5引入MTE搬运、UB中转和双缓冲后，每个Thread Block必须在多轮迭代间复用有限的UB缓冲区，这正是建立在"核内多轮循环"这个结构上（UB装不下全部256个band的数据）。所以这一步虽然本身是负收益，但它是后续所有优化的结构前提。

这个"本身不赚钱、但为后面铺路"的模式在本作业里会再出现一次（任务4），到时候可以回头对照一下。

## 5. 任务3：warp 分段规约优化访存形态

### 5.1 实现方案

任务1和任务2的核函数实现方式是同一个：**一个线程负责一个输出列，串行遍历自己的 `4 x 4` 窗口**，每个线程做 16次GM读 + 15次比较。本任务用SIMT的**warp规约**能力把这个过程改造掉，**仍然直接基于GM读写，不引入任何UB中转**——这样才能看清收益究竟来自哪里。

**关键约束：一个 `4 x 4` 窗口只有16个元素，而一个warp有32条lane。** 这个数量不匹配决定了该选哪个接口：

| 接口 | 分组能力 | 用在本算子上 |
| --- | --- | --- |
| `asc_reduce_max(val)` | **无 `width` 参数**，对warp内**所有活跃lane**规约成1个值 | 规约宽度16 ≠ warpSize 32。要让两个窗口在同一warp内互不干扰，只能靠**线程分叉**让两个半warp分别规约——而分叉在硬件上是串行执行的 |
| `asc_shfl_down(val, delta, width)` | `width` 参数把warp划分成**独立分组**，组内独立交换 | 取 `width = 4` 正好让4条相邻lane构成一组，对应一个输出的4个输入列，**无需分叉** |

所以本任务用 **`asc_shfl_down` 做分段规约**。`asc_reduce_max` 在本算子上的完整实测对比（含两种不同的实现方式）见7.3.2节——它不是"写得不够好"，而是接口语义与本算子的规约宽度不匹配。

**新的线程映射：一个线程负责一个输入列。**

`threadIdx.x` 不再是输出列，而是输入列号，取值 `0 ~ width-1`。因此每个Thread Block的线程数从 `out_width`（256）变为 `width`（1024），Host侧的 `dim3 block` 要改成 `in_width`，核函数的 `__launch_bounds__` 也要跟着改。

**核函数要做的事**，分两步：

1. **纵向规约**：每个线程求自己那一列 `window` 行的最大值，`4次读 + 3次比较`。读取地址的形式是 `input[(input_row_base + r) * width + in_col]`——同一warp内32条lane的 `in_col` 连续，这是本任务的关键。
2. **横向规约**：用两次 `asc_shfl_down(max_val, delta, window)` 把相邻4条lane的结果合并，`delta` 依次取 2、1（`log2(4) = 2` 步）。规约结果落在每个分组的第0条lane上，因此**只有 `in_col % window == 0` 的lane负责写回**，其余lane不写。

整体的数据流程如下（图中只画了8条lane = 2个分组，实际一个warp的32条lane构成8个分组）：

<img src="./images/07_07_maxpool/warp_shfl_reduce.png" alt="warp_shfl_reduce"  width="800px" >

图上省略了几处具体细节，这里补充说明：

**`delta` 的取值和方向。** `asc_shfl_down(v, delta, width)` 是**向低lane方向**传值：组内第 `i` 条lane拿到第 `i + delta` 条lane的 `v`。所以两步的写法是 `delta = 2`（lane0取lane2、lane1取lane3）然后 `delta = 1`（lane0取lane1），每次都和自己手上的值取较大者。分组宽度4需要 `log2(4) = 2` 步，这也是图中规约树只有两层的原因。

**超出分组范围的lane拿到什么。** 第一步之后，组内lane2、lane3手上的值已经没有意义（它们的 `i + delta` 落在分组之外）；第二步之后只有lane0持有完整结果。这些lane并不会出错，只是结果无效、不参与写回，所以图里没有画它们后续的连线。

**写回的判断条件和地址。** 每个分组的第0条lane就是 `in_col % window == 0` 的那条，它写 `out_band` 的第 `in_col / window` 个位置。其余lane直接跳过，不做任何写操作。

**分组是硬件行为，不是分叉。** `width=4` 让8个分组在**同一条指令**内并行完成各自的规约，全部32条lane始终活跃。

grid-stride循环结构与任务2完全一致，不需要改。`asc_shfl_down` 由 `simt_api/asc_simt.h` 提供（框架已包含），无需额外引入。

**为什么这样会更快？** 关键不在"比较次数变少了"——实际上总的GM元素读取次数完全没变（都是 `1024 x 1024 = 1M` 次），warp指令数反而上升了。真正的变化是**访存模式**：

| 版本 | 同一warp的32条lane读什么 | 一条读指令跨越 |
| --- | --- | --- |
| 任务1/2（一线程一输出列） | `out_col * WINDOW` 步长为4，地址两两相隔4个 `float` | **128个 `float` = 512字节**，命中4段 |
| 任务3（一线程一输入列） | `in_col` 连续，地址是连续的32个 `float` | **32个 `float` = 128字节**，命中1段 |

在SIMT模型里，一个warp的一条读指令会被硬件合并成尽可能少的访存事务。lane地址连续时一条指令只需1个事务；地址分散时同一条指令要拆成多个事务，延迟成倍增加。**这是SIMT编程中最基础也最容易被忽略的优化点：不改变算法、不减少计算量，只调整"哪个线程读哪个数据"。**

**请在运行前先预测**：读取次数没减少、warp指令数还变多了，你认为耗时会降多少？

### 5.2 编写并编译运行

框架代码与任务1/2相同（`Sources/07.07/maxpool/task3_gm_warp_shfl/maxpool_simt.asc`）。注意本任务要同时改核函数、`__launch_bounds__` 和Host侧的 `dim3 block`——三处必须一致，否则结果会错。

In [2]:
!cd Sources/07.07/maxpool/task3_gm_warp_shfl && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

/bin/bash: line 1: cd: Sources/07.07/maxpool/task3_gm_warp_shfl: No such file or directory


### 5.3 参考答案

In [3]:
!cat answer/07_07_maxpool/gm_warp_shfl/maxpool_gm_warp_shfl.asc

#include <algorithm>
#include <cmath>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"
#include "simt_api/asc_simt.h"

constexpr uint32_t WINDOW = 4;            // 池化窗口/步长，4x4 不重叠
constexpr uint32_t MAX_THREAD_COUNT = 1024; // 一个 Thread Block 覆盖输入矩阵的一行（一个线程一个输入列）

// 任务3 参考实现（纯SIMT 编程 + GM 直读 + warp 分段规约）：在任务2 的基础上只改线程映射与规约方式。
// 任务1/2 是"一个线程负责一个输出列"，串行遍历自己的 4x4 窗口（16 次读 + 15 次比较）；
// 本版本改成"一个线程负责一个输入列"（启动线程数从 out_width=256 变为 width=1024），
// 每个线程只纵向规约自己这一列的 window 个元素（4 次读 + 3 次比较），
// 再用 width=window 的分段 asc_shfl_down 把相邻 4 条 lane 的结果横向合并成 1 个输出。
// 关键收益是访存形态：同一 warp 的 32 条 lane 读的是 GM 上连续的 32 个 float（128B，1 段），
// 而任务1/2 同一 warp 的 32 条 lane 跨越 128 个 float（512B，4 段）。
// 实测 7.65us -> 4.9us（快约 36%），是本作业中单步收益最大的一项优化。
template <uint32_t window>
__global__ __launch_bounds__(MAX_THREAD_COUNT) void maxpool_gm_warp_shfl_custom(
    float* output, const float* input, uint32_t width, uint32_t height, uint32_t total_bands)
{
    uint32_t out_width = width / window;

#### 5.3.1 性能数据采集与分析

请测试性能数据分析优化效果。

以下为Ascend 950 环境实测（`msopprof` Task Duration，多次采集，64物理核）的参考结果：

| 实现 | Task Duration(μs) | 相对任务2 |
| --- | --- | --- |
| 任务1（SIMT编程，直接GM，256 Block） | 13.00 | — |
| 任务2（SIMT编程，直接GM，64 Block） | 14.04 | 基准 |
| 任务3（warp 分段规约，一线程一输入列） | **7.7 ~ 8.0** | **快约 45%** |


## 6. 任务4：混合编程，MTE搬运 + UB中转（单缓冲）

### 6.1 实现方案

前三个任务都是纯SIMT编程、直接基于GM读写。任务3已经把**计算侧**（线程映射与访存形态）调到了合理状态。本任务开始切换到 **SIMD/SIMT混合编程**模式，优化**数据侧**：不再让SIMT线程直接访问GM，而是**先由MTE单元把数据批量搬入UB，SIMT VF只负责UB内的规约**。

MaxPool按band切分后，GM上的读写本来就是连续的、且窗口不重叠导致零复用，所以引入UB中转并不能带来数据复用上的好处。那为什么还要做？两个原因：

1. **实际算子很少是孤立的**。如果MaxPool要和其他算子做流水衔接（结果留在UB上直接给下一段计算用），UB中转是必需的结构。
2. **搬运和计算可以解耦**，从而进一步做流水重叠——这是任务5的基础。

但引入UB中转有个前提：**关键在于由谁来搬**。让每个SIMT线程逐元素手工搬运效率很低（本章末尾给出了实测：那样做要16.7μs，比直接读GM还慢一倍），而让MTE单元负责GM/UB之间的批量连续搬运效率高得多。本任务验证这个思路——**MTE单元负责搬运，SIMT VF只负责UB内的窗口规约计算**。

**注意换框架**：本任务起改用 `hybrid_skeleton`（`maxpool_hybrid.asc`），它的CMakeLists**不带** `--enable-simt`。核函数从 `__global__` 变成 `__global__ __vector__`，SIMT代码改为通过 `asc_vf_call` 调起的 `__simt_vf__` 函数。这一点是混合编程的基本结构，参见07.06节。

**整体结构**，参照07.06节3.3，共三个部分（框架里三个函数都是空的）：

1. **`copy_gm_2band_to_ub`**（`__aicore__`，MTE2搬运）：把本Thread Block分到的band从GM搬入UB输入缓冲区。一个band在GM上是 `BAND_ROWS * width` 个连续 `float`，所以这是最简单的连续搬运形式，用 `asc_copy_gm2ub_align(dst_ub, src_gm, 字节数)` 即可。注意循环处理 `BANDS_PER_BLOCK` 个band，并对超出 `total_bands` 的部分做边界判断。

2. **`maxpool_band_reduce`**（`__simt_vf__`，SIMT VF规约）：**沿用任务3的线程映射与规约方式**——一个线程负责一个输入列，纵向规约后用 `asc_shfl_down(width=WINDOW)` 横向合并。唯一的区别是数据源从GM指针变成了UB指针。线程组织为 `dim3(width, BANDS_PER_BLOCK, 1)`，即 `threadIdx.x` 是输入列、`threadIdx.y` 是本Thread Block内的第几个band。

3. **`copy_ub_2band_to_gm`**（`__aicore__`，MTE3搬运）：用 `asc_copy_ub2gm_align` 把规约结果搬回GM。每个band的输出是 `out_width` 个连续 `float`。

**核函数要做的事**

- 声明UB缓冲区：输入 `BANDS_PER_BLOCK` 个band（每个 `BAND_ROWS x 1024`），输出 `BANDS_PER_BLOCK x 256`。
- grid-stride循环：`band_base` 从 `block_idx * BANDS_PER_BLOCK` 开始，步长 `block_num * BANDS_PER_BLOCK`。
- 循环体内按 **MTE2搬入 → SIMT VF规约 → MTE3搬出** 的顺序，用 `asc_lock`/`asc_unlock` 约束 `PIPE_MTE2` → `PIPE_V` → `PIPE_MTE3` 三者的执行次序（三者之间存在数据依赖，必须加锁）。本任务是**单缓冲**：只有一组缓冲区、一组mutex，三条流水线因此必然串行。

**一个容易错的地方**：UB中band的行跨距是 `width`（`1024`），不是 `out_width`。定位元素时用 `band[r * width + in_col]`。

**请在运行前先预测**：任务3已经把访存改成了完全合并的形态，现在再引入MTE + UB中转，你认为还能再快多少？

### 6.2 编写并编译运行

框架代码在 `Sources/07.07/maxpool/task4_simd_simt_ub/maxpool_hybrid.asc`。执行下面的cell查看这份混合编程框架（和前三个任务的框架不同）：

In [ ]:
!cat Sources/07.07/maxpool/task4_simd_simt_ub/maxpool_hybrid.asc

任务5用的是同一份混合编程框架，因此这份代码也是任务5的起点。实现后执行下面的单元格编译运行：

In [ ]:
!cd Sources/07.07/maxpool/task4_simd_simt_ub && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

### 6.3 参考答案

In [ ]:
!cat answer/07_07_maxpool/simd_simt_ub/maxpool_simd_simt_ub.asc

#### 6.3.1 性能数据采集与分析

请实测性能数据并完成性能分析。

以下为参考性能数据及分析：

| 实现 | Task Duration(μs) | 相对任务3 |
| --- | --- | --- |
| 任务3（纯SIMT，直接GM，shfl规约） | 8.0 | 基准 |
| 任务4（混合编程，MTE + UB单缓冲） | 7.7~8.9 | **基本持平** |

`msopprof` 的流水分项数据如下：

| 实现 | aiv_vec_time(μs) | aiv_scalar_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) | aiv_total_cycles |
| --- | --- | --- | --- | --- | --- |
| 任务3（直接GM） | 3.543 | 0.405 | 0.004 | 0.001 | 6533 |
| 任务4（MTE + UB单缓冲） | **1.188** | 0.483 | **2.563** | 0.409 | 7254 |

**`aiv_vec_time` 从 3.543μs 降到 1.188μs（降66%）——UB中转确实让计算侧快了很多**，因为VF现在读的是UB（片上，延迟低）而不是GM。但同时 **`aiv_mte2_time` 涨到了 2.563μs**。

而单缓冲版本的三条流水线是**串行**的（共用同一组缓冲区，被锁强制串行化），所以总耗时约等于三段之和：`1.188 + 2.563 + 0.409 ≈ 4.16μs` 的流水时间，加上scalar和其他开销，总cycle数（7254）比任务3（6533）还略高一点。

**这就是任务4的真正意义：它本身不是一个性能优化，而是一次结构重组。** 它把"GM访问"从vector流水里剥离出来、交给专职的MTE单元，为下一步的**流水重叠**创造了条件——三段耗时分离到不同的流水线上，才有可能让它们并行起来。这个价值要到任务5才兑现。


## 7. 任务5：混合编程 + 双缓冲

### 7.1 实现方案

任务4中，MTE2搬入、SIMT VF规约、MTE3搬出三个阶段因为共用同一组缓冲区而必须串行——这也是任务4相对任务3几乎没有收益的直接原因。本任务参照07.06节3.5引入ping/pong双缓冲，让三条流水线相互重叠，把任务4搭好的结构真正变成性能。

**要改的地方只在核函数内部。** 任务4实现的三个函数（`copy_gm_2band_to_ub`、`maxpool_band_reduce`、`copy_ub_2band_to_gm`）**完全复用，一行都不用改**——双缓冲只改变核函数中缓冲区的组织和锁的管理方式。

**结构要点**

- **两组独立的输入/输出缓冲区**：把UB缓冲区的最外层维度开成2，用 `curr_buffer = loop_count & 1` 在迭代间轮换。注意UB总容量有限，两组缓冲区加起来不能超出。
- **两组独立的mutex**：`DB_INPUT_MUTEX_BASE + curr_buffer`、`DB_OUTPUT_MUTEX_BASE + curr_buffer`（框架里两个base常量已定义）。**用不同的mutex是双缓冲能重叠的关键**——如果两组缓冲区共用一个mutex，锁本身就会把它们重新串行化，双缓冲的结构就白搭了。
- **锁的调用序列**（每轮迭代内，注意都作用在 `curr_buffer` 那一组上）：

  ```cpp
  asc_lock(PIPE_MTE2, input_mutex);
  // 搬入 in_band[curr_buffer]
  asc_unlock(PIPE_MTE2, input_mutex);

  // SIMT VF 等待当前输入 buffer 搬入完成；
  // 复用输出 buffer 前还要等待它上一次的 MTE3 搬出完成
  asc_lock(PIPE_V, input_mutex);
  asc_lock(PIPE_V, output_mutex);
  // asc_vf_call<maxpool_band_reduce>(...)
  asc_unlock(PIPE_V, input_mutex);
  asc_unlock(PIPE_V, output_mutex);

  asc_lock(PIPE_MTE3, output_mutex);
  // 搬出 out_band[curr_buffer]
  asc_unlock(PIPE_MTE3, output_mutex);
  ```

- 循环体末尾递增 `loop_count`，让下一轮切到另一组缓冲区。

这样当前迭代的MTE2搬入使用一组缓冲区时，上一轮的SIMT VF规约和MTE3搬出可以在另一组缓冲区上并行执行。

**请在运行前先预测**：任务4的三段流水耗时分别约为 `MTE2 = 2.56μs`、`VF = 1.19μs`、`MTE3 = 0.41μs`。如果三段能完美重叠，总耗时会趋近于最长的一段。据此估算双缓冲能带来多少收益。

### 7.2 编写并编译运行

框架代码与任务4相同（`Sources/07.07/maxpool/task5_simd_simt_ub_db/maxpool_hybrid.asc`）。建议直接把任务4的实现拷过来，再把缓冲区和锁改成双缓冲结构。

In [ ]:
!cd Sources/07.07/maxpool/task5_simd_simt_ub_db && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

### 7.3 参考答案

In [ ]:
!cat answer/07_07_maxpool/simd_simt_ub_db/maxpool_simd_simt_ub_db.asc

#### 7.3.1 性能数据采集与分析

请完成性能数据采集与分析，以下为实测数据参考分析：

| 实现 | Task Duration(μs) | 相对任务4 |
| --- | --- | --- |
| 任务4（混合编程，单缓冲） | 7.7 ~ 8.9 | 基准 |
| 任务5（混合编程，双缓冲） | **6.85** | **快约 15%** |

双缓冲把任务4搭好的结构变成了性能。`msopprof` 的流水分项数据说明了机制：

| 实现 | aiv_vec_time(μs) | aiv_scalar_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) | aiv_total_cycles |
| --- | --- | --- | --- | --- | --- |
| 任务4（单缓冲） | 1.188 | 0.483 | 2.563 | 0.409 | 7254 |
| 任务5（双缓冲） | 1.093 | 0.517 | 2.610 | 0.405 | **6433** |

注意**三段流水各自的耗时几乎完全没变**（搬运的数据量、规约的计算量一个字节都没变），变化的只有总cycle数：7254 → 6433。这正是流水重叠的特征——**不是让每一段变快，而是让它们同时进行**。

收益幅度（15%）比07.06节Transpose上的双缓冲要小，原因值得想清楚：双缓冲的收益来自"把串行的三段流水压缩到接近其中最长的一段"，因此**三段耗时越接近、重叠的收益越大**。MaxPool的三段严重不均衡——MTE2要搬入 `4 * 1024 * 2` 个 `float`（32KB），而MTE3只搬出 `256 * 2` 个 `float`（2KB），相差16倍；VF只占1.09μs。理论下界约等于最长的一段（MTE2 的 2.61μs）加上无法重叠的固定开销，重叠掉MTE3和VF的时间对总耗时的改善自然有限。

**现在的瓶颈在MTE2搬入。** 任务5之后 `aiv_vec_time`（1.09μs）已经远小于 `aiv_mte2_time`（2.61μs），整体耗时由MTE2主导。继续在VF上优化收益会很有限。


## 8. 作业提交要求

请提交以下内容：

**1. 五个任务的完整代码**

`Sources/07.07/maxpool/` 下五个任务目录中你实现的 `.asc` 文件，每个都要能编译通过并输出 `[Success] Case accuracy verification passed.`。

**2. 性能实测表格**

在你自己的环境上采集五个版本的耗时，填写下表（使用 `msopprof` 实测补充Task Duration数据；差距在10%以内的相邻版本，请补充host侧多次交替测量的均值）：

| 任务 | 编程模式 | 核函数 | 平均耗时(μs) |
| --- | --- | --- | --- |
| 任务1 | SIMT编程 | `maxpool_gm_naive_custom` | |
| 任务2 | SIMT编程 | `maxpool_gm_core_limited_custom` | |
| 任务3 | SIMT编程 | `maxpool_gm_warp_shfl_custom` | |
| 任务4 | 混合编程 | `maxpool_simd_simt_ub_custom` | |
| 任务5 | 混合编程 | `maxpool_simd_simt_ub_db_custom` | |

**3. 分析报告**

回答以下问题，每题不少于三句话，要求结合你的实测数据：

1. 任务2相对任务1限制了Thread Block数，为什么在MaxPool上几乎没有收益？如果把输入规模从 `1024 x 1024` 扩大到 `8192 x 8192`，你预期这一步的收益会变大还是变小？为什么？
2. 任务3为什么选 `asc_shfl_down` 而不是 `asc_reduce_max`？
3. 任务5的双缓冲带来约15% 的收益，而三段流水各自的耗时几乎完全没变。请解释收益是怎么产生的，并结合三条流水线的数据量估算为什么收益幅度不更大。
4. 如果把窗口从 `4 x 4`（不重叠）改成 `4 x 4` 但**步长为2**（相邻窗口重叠一半），你认为UB中转的收益会变大吗？为什么？（提示：想一想重叠窗口给数据复用带来了什么变化）

### 8.1 参考答案

完成上面的分析报告后，执行下面的单元格查看参考答案。答案文件包含五个任务的
参考性能数据（Task Duration 与 `msopprof` 流水分项）、四道分析题的完整解答，
以及几个容易被忽略的补充结论。

各任务代码的参考实现见前面各节的"参考答案"单元格。

In [ ]:
!cat ./answer/07.07_answer.txt